# §11.4.4 — 병목 채널 수에 따른 계산량–성능 곡선

> 딥러닝 교재 · 3부 11장 4절 4항 (🐍)
> 선행: §11.4.1($1\times1$ = 픽셀별 선형) · §11.4.2(병목의 계산량) · §11.4.3(손계산)

## 이 노트북이 답하는 질문

1. **병목을 얼마나 좁혀도 되는가?** 과제가 요구하는 채널 정보량 아래로 내려가면 무슨 일이 생기는가.
2. **임계 폭은 과제에서 예측 가능한가?** 필요한 선형 결합의 수를 알고 설계한 과제로 확인한다.
3. **같은 계산 예산이라면** 병목으로 아낀 계산을 깊이에 재투자하는 것이 나은가.

**예상 실행 시간** CPU 약 2분 (`FAST = True`이면 약 40초).
이 실험을 이해하려면 §11.4.2의 계산량 산수가 필요합니다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제 — 필요한 채널 정보량을 우리가 아는 문제

입력은 $12$채널 $8\times8$. 앞의 여섯 채널이 **세 쌍** $(0,1),(2,3),(4,5)$을 이루고,
무작위로 고른 한 쌍의 같은 위치에 십자 무늬가 찍힌다.
부호가 **정렬**(+,+)이면 클래스 1, **반정렬**(+,−)이면 클래스 0. 나머지 여섯 채널은 잡음뿐이다.

판별에는 각 쌍의 합/차 여섯 개의 선형 결합이 필요하도록 설계했다.
**병목 폭 $r$이 6 아래로 내려가면 정보가 통과하지 못할 것**이 설계에서 예측된다.

In [ ]:
C0, IMG = 12, 8
PAT = np.zeros((3, 3)); PAT[1, :] = 1.0; PAT[:, 1] = 1.0     # 십자
AMP = 3.0

def make_data(n, rn):
    X = rn.standard_normal((n, C0, IMG, IMG))
    y = rn.integers(0, 2, n)
    pair = rn.integers(0, 3, n)
    r0 = rn.integers(0, IMG-2, n); c0 = rn.integers(0, IMG-2, n)
    sgn = rn.choice([-1., 1.], n)
    for i in range(n):
        ca, cb = 2*pair[i], 2*pair[i]+1
        s2 = sgn[i] if y[i] else -sgn[i]
        X[i, ca, r0[i]:r0[i]+3, c0[i]:c0[i]+3] += AMP * sgn[i] * PAT
        X[i, cb, r0[i]:r0[i]+3, c0[i]:c0[i]+3] += AMP * s2 * PAT
    return X, y.astype(float)

Xte, yte = make_data(3000, np.random.default_rng(SEED + 99))
print("시험 표본:", Xte.shape)

---
## 2. 병목 망 — $1\times1(12\to r)$ → ReLU → $3\times3(r\to8)$ → 풀링 → 선형

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def im2col_c(X, k=3):
    # X: (N,C,H,W) -> (N, Ho, Wo, C*k*k)
    v = sliding_window_view(X, (k, k), axis=(2, 3))       # (N,C,Ho,Wo,k,k)
    return np.ascontiguousarray(v.transpose(0, 2, 3, 1, 4, 5)).reshape(
        X.shape[0], X.shape[2]-k+1, X.shape[3]-k+1, -1)

def sigmoid(z):
    return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))

def adam(p, g, m, v, t, lr):
    m[:] = 0.9*m + 0.1*g; v[:] = 0.999*v + 0.001*g*g
    p -= lr*(m/(1-0.9**t))/(np.sqrt(v/(1-0.999**t))+1e-8)

class BottleneckNet:
    # [1x1(C0->r) -> ReLU -> 3x3(r->F) -> ReLU] x n_blocks -> 풀링 -> 선형
    def __init__(self, r, F=8, blocks=1, rn=None):
        rn = rn or np.random.default_rng(0)
        self.blocks = []
        cin = C0
        for _ in range(blocks):
            W1 = rn.standard_normal((cin, r)) * np.sqrt(2/cin)
            b1 = np.zeros(r)
            W2 = rn.standard_normal((r*9, F)) * np.sqrt(2/(r*9))
            b2 = np.zeros(F)
            self.blocks.append([W1, b1, W2, b2])
            cin = F
        self.F = F
        self.u = rn.standard_normal(2*F) / np.sqrt(2*F)
        self.c = np.zeros(1)
        self.params = [p for blk in self.blocks for p in blk] + [self.u, self.c]
    def macs_per_pos(self):
        m, cin = 0, C0
        for (W1, b1, W2, b2) in self.blocks:
            r = W1.shape[1]; F = W2.shape[1]
            m += cin*r + r*9*F
            cin = F
        return m
    def forward(self, X):
        self.cache = []
        A = X
        for (W1, b1, W2, b2) in self.blocks:
            N, C, H, Wd = A.shape
            Z1 = np.einsum('nchw,cr->nrhw', A, W1, optimize=True) + b1[None, :, None, None]
            A1 = np.maximum(Z1, 0)
            col = im2col_c(A1)                        # (N,Ho,Wo,r*9)
            Z2 = col @ W2 + b2                        # (N,Ho,Wo,F)
            A2 = np.maximum(Z2, 0).transpose(0, 3, 1, 2)  # (N,F,Ho,Wo)
            self.cache.append((A, Z1, A1, col, Z2, A2.shape))
            A = A2
        self.Alast = A
        Amax = A.max(axis=(2, 3)); Aavg = A.mean(axis=(2, 3))
        feat = np.concatenate([Amax, Aavg], axis=1)
        self.feat = feat
        return feat @ self.u + self.c
    def backward(self, dout, X):
        gs = []
        du = self.feat.T @ dout; dc = np.array([dout.sum()])
        F = self.F
        dfeat = np.outer(dout, self.u)
        A = self.Alast
        N, _, Ho, Wo = A.shape
        dA = np.zeros_like(A)
        dA += dfeat[:, F:, None, None] / (Ho*Wo)
        flat = A.reshape(N, F, -1)
        idx = flat.argmax(axis=2)
        dflat = np.zeros_like(flat)
        n_i = np.arange(N)[:, None]; f_i = np.arange(F)[None, :]
        dflat[n_i, f_i, idx] += dfeat[:, :F]
        dA += dflat.reshape(A.shape)
        for bi in range(len(self.blocks)-1, -1, -1):
            W1, b1, W2, b2 = self.blocks[bi]
            Ain, Z1, A1, col, Z2, outshape = self.cache[bi]
            dA2 = dA.transpose(0, 2, 3, 1)             # (N,Ho,Wo,F)
            dZ2 = dA2 * (Z2 > 0)
            dW2 = col.reshape(-1, col.shape[-1]).T @ dZ2.reshape(-1, dZ2.shape[-1])
            db2 = dZ2.sum(axis=(0, 1, 2))
            dcol = dZ2 @ W2.T                          # (N,Ho,Wo,r*9)
            r = W1.shape[1]
            NN, HoW, WoW, _ = dcol.shape
            d6 = dcol.reshape(NN, HoW, WoW, r, 3, 3)
            dA1 = np.zeros_like(A1)
            for i in range(3):
                for j in range(3):
                    dA1[:, :, i:i+HoW, j:j+WoW] += d6[:, :, :, :, i, j].transpose(0, 3, 1, 2)
            dZ1 = dA1 * (Z1 > 0)
            dW1 = np.einsum('nchw,nrhw->cr', Ain, dZ1, optimize=True)
            db1 = dZ1.sum(axis=(0, 2, 3))
            dA = np.einsum('nrhw,cr->nchw', dZ1, W1, optimize=True)
            gs = [dW1, db1, dW2, db2] + gs
        return gs + [du, dc]

def train_eval(net, Xtr, ytr, Xev, yev, steps, seed=0, lr=4e-3, batch=128):
    ms = [np.zeros_like(p) for p in net.params]
    vs = [np.zeros_like(p) for p in net.params]
    rb = np.random.default_rng(seed)
    for t in range(1, steps+1):
        idx = rb.integers(0, len(ytr), batch)
        z = net.forward(Xtr[idx])
        p = sigmoid(z.ravel())
        dz = (p - ytr[idx]) / batch
        gs = net.backward(dz, Xtr[idx])
        for pp, g, m, v in zip(net.params, gs, ms, vs):
            adam(pp, g, m, v, t, lr)
    hits = 0
    for s0 in range(0, len(yev), 500):
        sl = slice(s0, s0+500)
        hits += np.sum((net.forward(Xev[sl]).ravel() > 0) == (yev[sl] > 0.5))
    return hits/len(yev)

---
## 3. 병목 폭 $r$을 훑는다

In [ ]:
RS = [1, 2, 3, 4, 6, 12]
SEEDS = 2 if FAST else 3
STEPS = 250 if FAST else 500
Xtr, ytr = make_data(4000, np.random.default_rng(SEED))
accs = np.zeros((len(RS), SEEDS)); macs = []
for ri, r in enumerate(RS):
    macs.append(BottleneckNet(r).macs_per_pos())
    for si in range(SEEDS):
        net = BottleneckNet(r, rn=np.random.default_rng(10*ri + si))
        accs[ri, si] = train_eval(net, Xtr, ytr, Xte, yte, STEPS, seed=si)
    print(f"r={r:2d}  MAC/pos={macs[-1]:4d}  acc={accs[ri].mean():.3f}±{accs[ri].std():.3f}  ({time.time()-_t0:.0f}초)")

---
## 4. 같은 예산 — 병목 깊은 망 vs 직접 얕은 망

직접 $3\times3(12\to12)$ 한 층의 위치당 MAC은 $12\cdot12\cdot9=1296$.
병목 블록 두 개($r=4$)는 그 절반 수준이다. 예산 대비 성능을 비교한다.

In [ ]:
class DirectNet(BottleneckNet):
    # 3x3(12->12) -> 풀링 -> 선형 (병목 없음), r=12를 1x1 항등처럼 쓰지 않고 직접 구현 대신
    # BottleneckNet(r=12, F=12, blocks=1)로 근사: 1x1(12->12)+3x3(12->12)
    pass

configs = {
    lab('직접 1블록', 'direct'): dict(r=12, F=12, blocks=1),
    lab('병목 1블록', 'bottleneck x1'): dict(r=4, F=8, blocks=1),
    lab('병목 2블록', 'bottleneck x2'): dict(r=4, F=8, blocks=2),
}
res = {}
for name, cfg in configs.items():
    a = []
    for si in range(SEEDS):
        net = BottleneckNet(cfg['r'], F=cfg['F'], blocks=cfg['blocks'],
                            rn=np.random.default_rng(100 + si))
        a.append(train_eval(net, Xtr, ytr, Xte, yte, STEPS, seed=50 + si))
    res[name] = (BottleneckNet(cfg['r'], F=cfg['F'], blocks=cfg['blocks']).macs_per_pos(),
                 np.mean(a), np.std(a))
    print(f"{name}: MAC/pos={res[name][0]}  acc={res[name][1]:.3f}±{res[name][2]:.3f}")

---
## 5. 교재 그림 — fig_11_4_4

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.7))

# (a) r 훑기: 정확도-계산량 곡선
ax = axes[0]
m = accs.mean(axis=1); sd = accs.std(axis=1)
ax.errorbar(macs, m, yerr=sd, fmt='o-', color=CB[5], ms=5, capsize=3)
for ri, r in enumerate(RS):
    ax.annotate(f'$r={r}$', (macs[ri], m[ri]),
                textcoords='offset points', xytext=(6, -11), fontsize=9)
ax.axvline(BottleneckNet(6).macs_per_pos(), color=CB[4], lw=0.8, ls=':')
ax.text(BottleneckNet(6).macs_per_pos()*1.03, 0.62,
        lab('설계 기준 폭 $r=6$', 'designed width $r=6$'), color=CB[4], fontsize=9)
ax.axhline(0.5, color='k', lw=0.6, ls=':')
ax.set_xlabel(lab('위치당 MAC (개)', 'MACs per position'))
ax.set_ylabel(lab('시험 정확도', 'test accuracy'))
ax.set_title(lab('(a) 병목 폭에 따른 계산량--성능 곡선', '(a) accuracy vs compute'), fontsize=10)

# (b) 예산--성능 평면에 모든 구성
ax = axes[1]
ax.errorbar(macs, m, yerr=sd, fmt='o-', color=CB[5], ms=4, capsize=2, alpha=0.9,
            label=lab('병목 폭 훑기 (1블록)', 'bottleneck sweep'))
names = list(res)
mks = ['s', 'D', '^']; cols = [CB[0], CB[1], CB[3]]
for i, k in enumerate(names):
    ax.errorbar(res[k][0], res[k][1], yerr=res[k][2], fmt=mks[i], color=cols[i],
                ms=8, capsize=3, label=k)
ax.set_xlabel(lab('위치당 MAC (개)', 'MACs per position'))
ax.set_ylabel(lab('시험 정확도', 'test accuracy'))
ax.set_title(lab('(b) 인수분해는 같은 정확도를 더 싸게 산다', '(b) all configs in the budget plane'), fontsize=10)
ax.legend(fontsize=8, loc='lower right')

save_book_fig(fig, 'fig_11_4_4')
plt.show()

> ### 읽는 법
>
> (a) 성능은 설계 기준 폭 $r=6$ 근처까지 완만히 오르고, 그 아래에서 뚜렷이 나빠진다.
> 절벽이 아니라 **완만한 붕괴**인 것도 정보다 — ReLU 이후의 특징은 선형 결합만이 아니어서
> 좁은 병목이 정보 일부를 비선형적으로 우겨 넣기 때문이다 (자기 점검 1).
> (b) $1\times1$ + $3\times3$ 인수분해(오른쪽 끝 파란 점)는 직접 $3\times3$과 같은 정확도를
> 약 70\% 예산으로, $r=6$ 병목은 소폭의 정확도를 내주고 35\% 예산으로 산다.
> 반면 **깊이 재투자(2블록)는 이 과제에서 이득이 없다** — 요구되는 변환이 얕기 때문이다.
> 병목의 이득도, 재투자의 이득도 과제의 함수라는 것까지가 이 실험의 결론이다.

---
## 6. 자기 점검

1. (a)의 임계가 정확히 6이 아니라 그 근처에서 완만하게 나타나는 이유는? (힌트: ReLU 이후의 표현은 선형 결합만이 아니다)
2. 잡음 채널 여섯 개를 없애면($C_0=6$) 임계 폭이 달라지는가?
3. §11.4.2의 비율 공식 $2/r + k^2/r^2$ (동일 채널 가정)과 이 실험의 MAC 수치가 다른 이유를 설명하라.
4. 이 과제에서 2블록이 1블록을 이기지 못했다. 깊이 재투자가 이득이 되려면 과제가 어떤 성질을 가져야 하는가? 그런 과제를 하나 설계해 보라.

## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `AMP` | 1절 | 3.0 | 무늬 대비 |
| 쌍의 수 | 1절 | 3 | 필요한 선형 결합 수 = 임계 폭이 바뀐다 |
| `RS`, `STEPS`, `SEEDS` | 3절 | — | 훑는 폭과 학습 길이 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")